# 1. Introduction

This notebook implements the systematic hyperparameter tuning workflow for the one-month stock-direction classifier after baseline evaluation and candidate selection are complete. The notebook deliberately follows the data and governance rules defined in the PRD: it loads prepared features, tunes the selected model candidates with time-aware validation, applies a single final test evaluation, and saves audit-ready outputs for downstream production handoff.

This first section frames the purpose of the notebook and identifies the inputs, outputs, and constraints that must remain in place throughout the run.

In [1]:
import json
import pickle
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
np.random.seed(42)


## 1.1 Notebook scope and assumptions

This markdown block defines the operating assumptions for the tuning run. It describes the expected upstream artifacts from steps 2, 3, and 4, the tiered candidate strategy, and the governance constraints that prevent leakage. It also clarifies which items are intentionally out of scope for this notebook, such as data collection, feature redesign, and production scheduling.

Expected inputs already exist before this code cell runs: the prepared dataset, baseline metrics files, and the selection comparison file. The corresponding code cell will validate those assets and surface any missing or schema-invalid inputs before tuning begins.

In [2]:
NOTEBOOK_DIR = Path.cwd().resolve()
project_markers = ["data", "1_fetch_data", "2_prepare_data", "3_evaluate_model", "4_select_model", "5_tune_model", "6_predict_data"]

PROJECT_ROOT = None
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if all((candidate / marker).exists() for marker in project_markers):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing folders data and 1_fetch_data..6_predict_data.")

NOTEBOOK_DIR = PROJECT_ROOT / "5_tune_model"
PREPARED_DATA_PATH = PROJECT_ROOT / "2_prepare_data" / "prepared_dataset.parquet"
OUTPUT_DIR = NOTEBOOK_DIR
BASELINE_DIR = PROJECT_ROOT / "3_evaluate_model"
SELECTION_PATH = PROJECT_ROOT / "4_select_model" / "model_comparison_baselines.json"

print(f"Project root   : {PROJECT_ROOT}")
print(f"Prepared data  : {PREPARED_DATA_PATH}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Baseline dir   : {BASELINE_DIR}")
print(f"Selection file : {SELECTION_PATH}")


Project root   : /workspace/projects/1-month-direction-classifier
Prepared data  : /workspace/projects/1-month-direction-classifier/2_prepare_data/prepared_dataset.parquet
Output dir     : /workspace/projects/1-month-direction-classifier/5_tune_model
Baseline dir   : /workspace/projects/1-month-direction-classifier/3_evaluate_model
Selection file : /workspace/projects/1-month-direction-classifier/4_select_model/model_comparison_baselines.json


# 2. Configuration and run policy

This section centralizes all notebook settings so a user can run the tuning workflow reproducibly without editing logic throughout the file. It includes project-relative paths, candidate toggles, search-method settings, budget presets, CV configuration, and output naming conventions. The notebook keeps a deterministic, explicit configuration contract at the top so all downstream decisions are traceable.

In [3]:
NOTEBOOK_DIR = Path.cwd().resolve()
project_markers = ["data", "1_fetch_data", "2_prepare_data", "3_evaluate_model", "4_select_model", "5_tune_model", "6_predict_data"]

PROJECT_ROOT = None
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if all((candidate / marker).exists() for marker in project_markers):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing folders data and 1_fetch_data..6_predict_data.")

NOTEBOOK_DIR = PROJECT_ROOT / "5_tune_model"
PREPARED_DATA_PATH = PROJECT_ROOT / "2_prepare_data" / "prepared_dataset.parquet"
OUTPUT_DIR = NOTEBOOK_DIR
BASELINE_DIR = PROJECT_ROOT / "3_evaluate_model"
SELECTION_PATH = PROJECT_ROOT / "4_select_model" / "model_comparison_baselines.json"

print(f"Project root   : {PROJECT_ROOT}")
print(f"Prepared data  : {PREPARED_DATA_PATH}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Baseline dir   : {BASELINE_DIR}")
print(f"Selection file : {SELECTION_PATH}")

Project root   : /workspace/projects/1-month-direction-classifier
Prepared data  : /workspace/projects/1-month-direction-classifier/2_prepare_data/prepared_dataset.parquet
Output dir     : /workspace/projects/1-month-direction-classifier/5_tune_model
Baseline dir   : /workspace/projects/1-month-direction-classifier/3_evaluate_model
Selection file : /workspace/projects/1-month-direction-classifier/4_select_model/model_comparison_baselines.json


## 2.1 Define project paths and run toggles

The code cell corresponding to this section should use the existing notebook configuration variables to define the project root, prepared data path, benchmark artifacts directory, selection file path, and the output directory for this tuning notebook. For this HGB-specific notebook, the active candidate list is fixed to `hist_gradient_boosting_baseline`.

Inputs: configuration variables or literals already defined by the user, including output directory names and the dedicated HGB model name.

Logic: establish a consistent set of project-relative paths and set the single active candidate for the run.

Outputs: a validated configuration object with paths and HGB candidate scope, ready for downstream ingestion and validation.


In [4]:
RUN_CONFIG = {
    "search_method": "randomized",           # "randomized" | "grid"
    "budget_preset": "medium",              # "small" | "medium" | "large"
    "cv_n_splits": 4,
    "scoring_metric": "f1_macro",
    "random_seed": 42,
    "candidate_order": ["hist_gradient_boosting_baseline"],
}

BUDGET_MAP = {
    "small": 20,
    "medium": 50,
    "large": 100,
}

RUN_METADATA = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "5_tune_model_hgb.ipynb",
    "project_root": str(PROJECT_ROOT),
    "prepared_data_path": str(PREPARED_DATA_PATH),
    "baseline_dir": str(BASELINE_DIR),
    "selection_path": str(SELECTION_PATH),
    "output_dir": str(OUTPUT_DIR),
    "run_config": RUN_CONFIG,
}
print(json.dumps({k: v for k, v in RUN_METADATA.items() if k != "run_config"}, indent=2))
print(RUN_CONFIG)
print(BUDGET_MAP)

{
  "timestamp_utc": "2026-09-13T16:26:51.989108+00:00",
  "notebook": "5_tune_model_hgb.ipynb",
  "project_root": "/workspace/projects/1-month-direction-classifier",
  "prepared_data_path": "/workspace/projects/1-month-direction-classifier/2_prepare_data/prepared_dataset.parquet",
  "baseline_dir": "/workspace/projects/1-month-direction-classifier/3_evaluate_model",
  "selection_path": "/workspace/projects/1-month-direction-classifier/4_select_model/model_comparison_baselines.json",
  "output_dir": "/workspace/projects/1-month-direction-classifier/5_tune_model"
}
{'search_method': 'randomized', 'budget_preset': 'medium', 'cv_n_splits': 4, 'scoring_metric': 'f1_macro', 'random_seed': 42, 'candidate_order': ['hist_gradient_boosting_baseline']}
{'small': 20, 'medium': 50, 'large': 100}


## 2.2 Set search method, budget presets, and CV controls

This cell defines the tuning policy for the run. It should capture the selected search method (`randomized` or `grid`), the budget preset (`small`, `medium`, `large`), the number of cross-validation splits, the scoring metric, the random seed, and the candidate-specific search budget behavior. It should also state when a finite grid may override the budget preset.

Inputs: the top-level configuration previously established for candidate scope and search behavior.

Logic: configure deterministic settings for reproducibility and derive the effective trials count or grid cardinality used in the hyperparameter search.

Outputs: a structured tuning policy dictionary that downstream model-search code can reference.

In [5]:
df_raw = pd.read_parquet(PREPARED_DATA_PATH)
if not pd.api.types.is_datetime64_any_dtype(df_raw["date"]):
    df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw = df_raw.sort_values(["date", "symbol"]).reset_index(drop=True)
print(f"Loaded prepared dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Date range: {df_raw['date'].min().date()} -> {df_raw['date'].max().date()}")


Loaded prepared dataset: 3,864 rows x 49 columns
Date range: 2021-06-28 -> 2026-08-13


## 2.3 Capture run metadata and validation baseline

This cell documents the notebook run configuration in a machine-readable metadata object. It should record the run timestamp, candidate queue, search method, budget preset, CV settings, output directory, and data source references. The metadata can later be embedded in output artifacts and used to confirm that a run was completed under the intended governance settings.

Inputs: configuration values from the earlier cells.

Logic: normalize and package the run metadata so it is consistent across all tuned candidate artifacts.

Outputs: a metadata object and a record of the exact run configuration for artifact serialization.

In [6]:
RAW_COLS = ["symbol", "date", "open", "high", "low", "close", "adj_close", "volume"]
FEATURE_COLS = [
    "ret_1d", "log_ret_1d", "ret_5d", "ret_10d", "ret_20d", "ret_60d",
    "log_ret_5d", "log_ret_20d", "tr_range", "true_range", "sma_5", "sma_20",
    "sma_50", "sma_200", "close_over_sma_20", "close_over_sma_50",
    "close_over_sma_200", "sma_20_slope_5d", "sma_50_slope_5d", "vol_ret_5d",
    "vol_ret_20d", "vol_ret_60d", "vol_tr_5d", "vol_tr_20d", "value_traded",
    "vol_ma_20", "vol_ma_60", "vol_rel_20", "value_traded_ma_20", "value_traded_rel_20",
    "rsi_14", "macd", "macd_signal", "macd_hist", "atr_14", "hh_20d", "ll_20d",
    "dist_from_20d_high", "dist_from_20d_low",
]
TARGET_COLS = ["fwd_ret_20d", "label"]
required = RAW_COLS + FEATURE_COLS + TARGET_COLS
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if not pd.api.types.is_datetime64_any_dtype(df_raw["date"]):
    raise ValueError(f"Column 'date' is not datetime-like: {df_raw['date'].dtype}")
valid_labels = {"buy", "hold", "sell"}
unexpected = set(df_raw["label"].dropna().unique()) - valid_labels
if unexpected:
    raise ValueError(f"Unexpected label values present: {sorted(unexpected)}")
print("Schema validation passed for prepared dataset.")
print(f"Observed labels: {sorted(df_raw['label'].dropna().unique().tolist())}")


Schema validation passed for prepared dataset.
Observed labels: ['buy', 'hold', 'sell']


# 3. Data ingestion and validation

This section prepares the notebook for tuning by loading the prepared dataset and validating the upstream files and expected schemas. The notebook must stop with a clear message if the required artifacts do not exist or are malformed. This is the checkpoint that prevents tuning from running on stale, partial, or incompatible data.

In [7]:
with open(SELECTION_PATH, "r", encoding="utf-8") as fh:
    selection_data = json.load(fh)

baseline_payload = {}
for candidate in RUN_CONFIG["candidate_order"]:
    path = BASELINE_DIR / f"{candidate}_metrics.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing baseline metrics for {candidate}: {path}")
    with path.open("r", encoding="utf-8") as fh:
        baseline_payload[candidate] = json.load(fh)

candidate_registry = []
for candidate in RUN_CONFIG["candidate_order"]:
    candidate_registry.append(
        {
            "candidate_name": candidate,
            "tier": "Tier-2",
            "required": False,
            "baseline_metrics_path": str(BASELINE_DIR / f"{candidate}_metrics.json"),
            "selection_status": "selected_for_tuning" if candidate in selection_data.get("tuning_candidates", []) else "configured",
        }
    )

print("Candidate registry:")
for item in candidate_registry:
    print(f"  - {item['candidate_name']} ({item['tier']})")


Candidate registry:
  - hist_gradient_boosting_baseline (Tier-2)


## 3.1 Load prepared dataset and required baseline files

This cell loads the prepared parquet dataset from step 2 and the relevant baseline metrics JSON files from step 3. It also loads the model comparison JSON generated by step 4 so the notebook can align its tuning targets with the recommended candidate set and governance selection logic.

Inputs: `prepared_dataset.parquet`, baseline metrics files for the active model candidates, and `model_comparison_baselines.json`.

Logic: read the files, confirm they are present, and prepare them for schema validation and downstream comparison.

Outputs: loaded DataFrame and JSON objects representing the prepared data and the baseline/reference artifacts.

In [8]:
feature_cols = [c for c in FEATURE_COLS if c in df_raw.columns]
model_df = df_raw[["date", "symbol", *feature_cols, "label"]].dropna(subset=["label"]).copy()
model_df = model_df.sort_values("date").reset_index(drop=True)

n_rows = len(model_df)
train_cut = int(n_rows * 0.70)
val_cut = int(n_rows * 0.85)

train_df = model_df.iloc[:train_cut].copy().reset_index(drop=True)
val_df = model_df.iloc[train_cut:val_cut].copy().reset_index(drop=True)
test_df = model_df.iloc[val_cut:].copy().reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df["label"]
X_val = val_df[feature_cols]
y_val = val_df["label"]
X_test = test_df[feature_cols]
y_test = test_df["label"]

print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,} | Test rows: {len(test_df):,}")
print(f"Train date range: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"Val date range:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"Test date range:  {test_df['date'].min()} -> {test_df['date'].max()}")


Train rows: 2,704 | Val rows: 580 | Test rows: 580
Train date range: 2021-06-28 00:00:00 -> 2025-01-29 00:00:00
Val date range:   2025-01-29 00:00:00 -> 2025-11-04 00:00:00
Test date range:  2025-11-04 00:00:00 -> 2026-08-13 00:00:00


## 3.2 Validate schema, required columns, and label set

This cell validates the expected structure of the prepared dataset and the baseline artifact schema. It should confirm the presence of the required feature columns, ensure the label column contains the expected classes (`buy`, `hold`, `sell`), and verify that the baseline metrics JSON contains the keys needed for comparison and governance checks.

Inputs: prepared dataset and loaded baseline JSON objects.

Logic: validate required columns, label set, required metric names, and the candidate IDs referenced by the selection file. Raise actionable errors if any checks fail.

Outputs: an approved data-contract state that can proceed to time-aware splitting and model tuning.

In [9]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_tv = train_val_df[feature_cols]
y_tv = train_val_df["label"]

cv_splitter = TimeSeriesSplit(n_splits=RUN_CONFIG["cv_n_splits"])
cv_fold_summary = []
for fold_number, (train_idx, val_idx) in enumerate(cv_splitter.split(X_tv), start=1):
    train_dates = train_val_df.iloc[train_idx]["date"]
    val_dates = train_val_df.iloc[val_idx]["date"]
    cv_fold_summary.append(
        {
            "fold": fold_number,
            "train_start": train_dates.min().date().isoformat(),
            "train_end": train_dates.max().date().isoformat(),
            "val_start": val_dates.min().date().isoformat(),
            "val_end": val_dates.max().date().isoformat(),
            "train_rows": int(len(train_idx)),
            "val_rows": int(len(val_idx)),
        }
    )

print(f"TimeSeriesSplit configured with {len(cv_fold_summary)} folds.")
print(json.dumps(cv_fold_summary, indent=2))


TimeSeriesSplit configured with 4 folds.
[
  {
    "fold": 1,
    "train_start": "2021-06-28",
    "train_end": "2022-05-10",
    "val_start": "2022-05-11",
    "val_end": "2023-03-24",
    "train_rows": 660,
    "val_rows": 656
  },
  {
    "fold": 2,
    "train_start": "2021-06-28",
    "train_end": "2023-03-24",
    "val_start": "2023-03-24",
    "val_end": "2024-02-07",
    "train_rows": 1316,
    "val_rows": 656
  },
  {
    "fold": 3,
    "train_start": "2021-06-28",
    "train_end": "2024-02-07",
    "val_start": "2024-02-07",
    "val_end": "2024-12-18",
    "train_rows": 1972,
    "val_rows": 656
  },
  {
    "fold": 4,
    "train_start": "2021-06-28",
    "train_end": "2024-12-18",
    "val_start": "2024-12-19",
    "val_end": "2025-11-04",
    "train_rows": 2628,
    "val_rows": 656
  }
]


## 3.3 Confirm candidate recommendations and impose tiered scope

This cell resolves the active tuning candidate based on the selection guidance and the dedicated notebook policy. In this notebook, `hist_gradient_boosting_baseline` is the single active tuning target and is executed as the Tier-2 exploratory fallback path documented in the PRD.

Inputs: the selection JSON and the fixed HGB candidate configuration from the top block.

Logic: build the final candidate list and assign a tier and rationale for the HGB candidate.

Outputs: a candidate registry used by the search engine and the artifact metadata.


In [10]:
if "MODEL_NAME" not in globals():
    MODEL_NAME = "hist_gradient_boosting_baseline"
if "RUN_CONFIG" not in globals():
    RUN_CONFIG = {"search_method": "randomized", "budget_preset": "medium", "scoring_metric": "f1_macro"}
if "BUDGET_MAP" not in globals():
    BUDGET_MAP = {"small": 20, "medium": 50, "large": 100}
HGB_PARAM_SPACE = {
    "model__learning_rate": [0.01, 0.02, 0.05, 0.1],
    "model__max_iter": [200, 300, 500],
    "model__max_depth": [3, 5, 6, 8, None],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [10, 20, 40],
    "model__l2_regularization": [0.0, 0.1, 1.0],
    "model__early_stopping": [True],
}
SEARCH_CONFIG = {
    MODEL_NAME: {
        "param_space": HGB_PARAM_SPACE,
        "search_type": RUN_CONFIG["search_method"],
        "budget": BUDGET_MAP[RUN_CONFIG["budget_preset"]],
        "n_jobs": -1,
        "scoring": RUN_CONFIG["scoring_metric"],
        "refit": True,
    }
}
print(json.dumps({k: {"search_type": v["search_type"], "budget": v["budget"], "scoring": v["scoring"]} for k, v in SEARCH_CONFIG.items()}, indent=2))


{
  "hist_gradient_boosting_baseline": {
    "search_type": "randomized",
    "budget": 50,
    "scoring": "f1_macro"
  }
}


# 4. Split design and time-aware validation

This section defines the chronological data split strategy for tuning. The notebook must preserve the train/validation/test integrity described in the PRD and must ensure that all hyperparameter selection happens only on the train-plus-validation portion. It also records fold boundaries for transparency and governance.

In [11]:
hgb_pipeline = Pipeline([
    ("model", HistGradientBoostingClassifier(random_state=RUN_CONFIG["random_seed"], loss="log_loss")),
])
hgb_search = RandomizedSearchCV(
    estimator=hgb_pipeline,
    param_distributions=HGB_PARAM_SPACE,
    n_iter=BUDGET_MAP[RUN_CONFIG["budget_preset"]],
    cv=cv_splitter,
    scoring=RUN_CONFIG["scoring_metric"],
    refit=True,
    n_jobs=-1,
    random_state=RUN_CONFIG["random_seed"],
)
hgb_search.fit(X_tv, y_tv)

hgb_results = {
    "candidate": MODEL_NAME,
    "search": hgb_search,
    "best_score": float(hgb_search.best_score_),
    "best_params": hgb_search.best_params_,
    "cv_results": pd.DataFrame(hgb_search.cv_results_),
}
print(f"Best HGB CV score: {hgb_results['best_score']:.4f}")
print(hgb_results["best_params"])


Best HGB CV score: 0.3670
{'model__min_samples_leaf': 10, 'model__max_leaf_nodes': 63, 'model__max_iter': 500, 'model__max_depth': 3, 'model__learning_rate': 0.1, 'model__l2_regularization': 0.0, 'model__early_stopping': True}


## 4.1 Build train, validation, and test partitions

This code cell creates the chronological data split with the correct hold-out test set untouched until the final evaluation step. It should explicitly separate feature columns from the label target and keep the split boundaries reproducible and time ordered.

Inputs: the prepared dataset and the configured train/validation/test split strategy.

Logic: establish `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, and `y_test` with a chronological ordering and no shuffling.

Outputs: clean split objects that maintain leakage prevention for all tuning decisions.

In [12]:
search_results_by_candidate = {MODEL_NAME: hgb_results}
print(f"Active tuning candidate: {MODEL_NAME}")
print("Search object prepared for final export.")


Active tuning candidate: hist_gradient_boosting_baseline
Search object prepared for final export.


## 4.2 Define `TimeSeriesSplit` CV and audit fold boundaries

This cell defines the cross-validation strategy used during search. The code should construct a `TimeSeriesSplit` over the combined train+validation subset only, with a default `n_splits=4` and optional `n_splits=5` for larger budgets. It should also record fold start and end dates or row boundaries so the validation flow can be audited.

Inputs: train and validation splits, plus the configured CV settings.

Logic: create the CV iterator and log the fold date boundaries and fold counts in a traceable structure.

Outputs: a CV object and a fold-summary record used in the tuning search results.

In [13]:
for candidate, result in search_results_by_candidate.items():
    results_path = OUTPUT_DIR / f"{candidate}_tuning_search_results.csv"
    result["cv_results"].to_csv(results_path, index=False)
    result["results_path"] = str(results_path)
    print(f"Saved search results: {results_path}")


Saved search results: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuning_search_results.csv


# 5. Search strategy and parameter spaces

This section defines the model-search engine and the parameter grids or distributions used by each candidate. The PRD defines a mandatory Tier-1 MLP search and an optional Tier-2 HistGradientBoosting search. This section keeps the search spaces explicit, constrained, and aligned with the projectâ€™s governance and runtime constraints.

In [14]:
candidate_results = {}
for candidate, result in search_results_by_candidate.items():
    search = result["search"]
    final_model = clone(search.best_estimator_)
    X_train_val = pd.concat([train_df, val_df], ignore_index=True)[feature_cols]
    y_train_val = pd.concat([train_df, val_df], ignore_index=True)["label"]
    final_model.fit(X_train_val, y_train_val)
    candidate_results[candidate] = {
        "search": search,
        "best_params": search.best_params_,
        "best_score": result["best_score"],
        "final_model": final_model,
        "train_val_rows": len(X_train_val),
    }
    print(f"Refit complete for {candidate} on train+validation data.")


Refit complete for hist_gradient_boosting_baseline on train+validation data.


## 5.1 Define the `HistGradientBoostingClassifier` parameter space

This cell enumerates the hyperparameter search space for the `HistGradientBoostingClassifier` candidate. It follows the dedicated HGB tuning scope and includes the configured values for `learning_rate`, `max_iter`, `max_depth`, `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`, and `early_stopping`.

Inputs: model identity and candidate registry.

Logic: define the exact parameter space and state the search strategy used for this candidate.

Outputs: an HGB search specification object ready for `RandomizedSearchCV` or `GridSearchCV`.


In [15]:
LABEL_ORDER = globals().get("LABEL_ORDER", ["buy", "hold", "sell"])

tuned_test_payloads = {}
for candidate, result in candidate_results.items():
    model = result["final_model"]
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, labels=LABEL_ORDER, output_dict=True, zero_division=0)
    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
        "f1_buy": float(report["buy"]["f1-score"]),
        "f1_hold": float(report["hold"]["f1-score"]),
        "f1_sell": float(report["sell"]["f1-score"]),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=LABEL_ORDER).tolist(),
    }
    pred_df = pd.DataFrame(
        {
            "date": test_df["date"].astype(str).values,
            "symbol": test_df["symbol"].values,
            "actual_label": y_test.reset_index(drop=True).tolist(),
            "predicted_label": y_pred.tolist(),
        }
    )
    tuned_test_payloads[candidate] = {
        "metrics": metrics,
        "predictions": pred_df,
        "classification_report": report,
        "y_pred": y_pred,
    }
    print(f"Final test evaluation complete for {candidate}: macro F1 = {metrics['macro_f1']:.4f}")

Final test evaluation complete for hist_gradient_boosting_baseline: macro F1 = 0.3525


## 5.2 Define constrained exploratory variants for `HistGradientBoostingClassifier`

This cell documents the constrained exploratory framing for the Tier-2 HGB path. The notebook remains explicit that this candidate is tuned as a fallback exploration path, while still enforcing the same leakage controls and governance criteria as other candidates.

Inputs: the candidate registry and the configured HGB search space.

Logic: keep the exploratory parameter space bounded and transparent for auditability.

Outputs: an HGB search specification object used in this notebook run.


In [16]:
for candidate, payload in tuned_test_payloads.items():
    model_path = OUTPUT_DIR / f"{candidate}_tuned_model.pkl"
    metrics_path = OUTPUT_DIR / f"{candidate}_tuned_metrics.json"
    pred_path = OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv"

    with model_path.open("wb") as fh:
        pickle.dump(candidate_results[candidate]["final_model"], fh)
    payload["predictions"].to_csv(pred_path, index=False)

    metrics_payload = {
        "model_name": candidate,
        "model_type": str(type(candidate_results[candidate]["final_model"].named_steps["model"]).__module__) + "." + type(candidate_results[candidate]["final_model"].named_steps["model"]).__name__,
        "search_method": RUN_CONFIG["search_method"],
        "search_budget": RUN_CONFIG["budget_preset"],
        "search_space": SEARCH_CONFIG.get(candidate, {}).get("param_space", {}),
        "best_params": candidate_results[candidate]["best_params"],
        "cv_results_summary": {
            "best_cv_score": float(candidate_results[candidate]["best_score"]),
            "n_trials": len(candidate_results[candidate]["search"].cv_results_["params"]),
        },
        "data": {
            "train_start": train_df["date"].min().date().isoformat(),
            "train_end": train_df["date"].max().date().isoformat(),
            "val_start": val_df["date"].min().date().isoformat(),
            "val_end": val_df["date"].max().date().isoformat(),
            "test_start": test_df["date"].min().date().isoformat(),
            "test_end": test_df["date"].max().date().isoformat(),
            "n_train": len(train_df),
            "n_val": len(val_df),
            "n_test": len(test_df),
        },
        "baseline_test_metrics": baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}),
        "tuned_val_metrics": {
            "macro_f1": float(candidate_results[candidate]["best_score"]),
        },
        "tuned_test_metrics": payload["metrics"],
        "comparison": {},
        "governance": {},
        "already_tuned": False,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    with metrics_path.open("w", encoding="utf-8") as fh:
        json.dump(metrics_payload, fh, indent=2)
    print(f"Artifacts saved for {candidate}: {model_path.name}, {metrics_path.name}, {pred_path.name}")


Artifacts saved for hist_gradient_boosting_baseline: hist_gradient_boosting_baseline_tuned_model.pkl, hist_gradient_boosting_baseline_tuned_metrics.json, hist_gradient_boosting_baseline_tuned_test_predictions.csv


## 5.3 Configure the search engine and fallback rules

This cell defines how the notebook chooses between `RandomizedSearchCV` and `GridSearchCV`, along with the fallback logic for cases where a grid search has lower cardinality than the selected preset budget. It also sets the scoring metric to `f1_macro`, ensures `refit=True`, and states the expected behavior for parallel execution where supported.

Inputs: the selected search method, candidate search spaces, and the budget preset.

Logic: derive the effective estimator search configuration, decide whether to sample or exhaust the grid, and document the final effective search parameters.

Outputs: a search configuration object per candidate, including effective trial count and search method.

In [17]:
comparison_rows = []
for candidate, payload in tuned_test_payloads.items():
    baseline = baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {})
    tuned = payload["metrics"]
    cov = candidate_results[candidate]["best_score"]
    row = {
        "model_name": candidate,
        "baseline_test_macro_f1": float(baseline.get("macro_f1", 0.0)),
        "tuned_test_macro_f1": float(tuned["macro_f1"]),
        "baseline_test_f1_buy": float(baseline.get("f1_buy", baseline.get("per_class", {}).get("buy", {}).get("f1", 0.0))),
        "tuned_test_f1_buy": float(tuned["f1_buy"]),
        "baseline_test_f1_sell": float(baseline.get("f1_sell", baseline.get("per_class", {}).get("sell", {}).get("f1", 0.0))),
        "tuned_test_f1_sell": float(tuned["f1_sell"]),
        "val_macro_f1": float(cov),
        "stability_delta": float(cov - tuned["macro_f1"]),
        "lift_macro_f1": float(tuned["macro_f1"] - baseline.get("macro_f1", 0.0)),
        "lift_f1_buy": float(tuned["f1_buy"] - float(baseline.get("f1_buy", baseline.get("per_class", {}).get("buy", {}).get("f1", 0.0)))),
        "lift_f1_sell": float(tuned["f1_sell"] - float(baseline.get("f1_sell", baseline.get("per_class", {}).get("sell", {}).get("f1", 0.0)))),
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = OUTPUT_DIR / "baseline_vs_tuned_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(comparison_df)


                        model_name  baseline_test_macro_f1  \
0  hist_gradient_boosting_baseline                  0.3688   

   tuned_test_macro_f1  baseline_test_f1_buy  tuned_test_f1_buy  \
0             0.352509                0.4661           0.322767   

   baseline_test_f1_sell  tuned_test_f1_sell  val_macro_f1  stability_delta  \
0                 0.3037            0.345745      0.366953         0.014444   

   lift_macro_f1  lift_f1_buy  lift_f1_sell  
0      -0.016291    -0.143333      0.042045  


# 6. Search execution and model tuning

This section runs the time-aware tuning loop for each active candidate. Model training must occur only on the training folds inside the time-ordered cross-validation process. It must avoid test-set leakage and log the results in a format that can be exported to a search-results CSV.

In [18]:
governance_results = {}
for candidate, payload in tuned_test_payloads.items():
    baseline = baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {})
    tuned = payload["metrics"]
    val_macro_f1 = float(candidate_results[candidate]["best_score"])
    flag_instability = (val_macro_f1 - tuned["macro_f1"]) > 0.05
    reasons = []
    if tuned["macro_f1"] <= float(baseline.get("macro_f1", 0.0)):
        reasons.append("fail_test_macro_f1")
    if tuned["f1_buy"] < 0.20:
        reasons.append("fail_test_f1_buy")
    if tuned["f1_sell"] < 0.15:
        reasons.append("fail_test_f1_sell")
    if flag_instability:
        reasons.append("flag_instability")
    passed = len(reasons) == 0
    governance_results[candidate] = {
        "passed": passed,
        "reasons": reasons,
        "flag_instability": bool(flag_instability),
        "val_macro_f1": val_macro_f1,
        "test_macro_f1": tuned["macro_f1"],
        "test_f1_buy": tuned["f1_buy"],
        "test_f1_sell": tuned["f1_sell"],
    }
    print(f"{candidate}: {'PASS' if passed else 'FAIL'} | reasons={reasons}")


hist_gradient_boosting_baseline: FAIL | reasons=['fail_test_macro_f1']


## 6.1 Tune the `hist_gradient_boosting_baseline` candidate

This code cell executes the HGB tuning workflow for the active candidate. It wraps the estimator with preprocessing, performs cross-validation search using the time-aware splitter, and collects all trial-level results necessary for later reporting and CSV export.

Inputs: feature data for the train+validation subset, the HGB search space, and the configured CV strategy.

Logic: run the search, select the best set of hyperparameters based on validation `f1_macro`, and retain the tuned estimator metadata and trial results.

Outputs: best HGB parameters, a fitted search object, and a search-results summary ready for export.


In [19]:
for candidate, result in candidate_results.items():
    metrics_path = OUTPUT_DIR / f"{candidate}_tuned_metrics.json"
    with metrics_path.open("r", encoding="utf-8") as fh:
        metric_doc = json.load(fh)
    metric_doc["comparison"] = {
        "baseline_test_macro_f1": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("macro_f1", 0.0)),
        "tuned_test_macro_f1": float(tuned_test_payloads[candidate]["metrics"]["macro_f1"]),
        "lift_test_macro_f1": float(tuned_test_payloads[candidate]["metrics"]["macro_f1"] - baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("macro_f1", 0.0)),
        "baseline_test_f1_buy": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("f1_buy", baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("per_class", {}).get("buy", {}).get("f1", 0.0))),
        "tuned_test_f1_buy": float(tuned_test_payloads[candidate]["metrics"]["f1_buy"]),
        "baseline_test_f1_sell": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("f1_sell", baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("per_class", {}).get("sell", {}).get("f1", 0.0))),
        "tuned_test_f1_sell": float(tuned_test_payloads[candidate]["metrics"]["f1_sell"]),
        "stability_delta": float(candidate_results[candidate]["best_score"] - tuned_test_payloads[candidate]["metrics"]["macro_f1"]),
        "flag_instability": bool((candidate_results[candidate]["best_score"] - tuned_test_payloads[candidate]["metrics"]["macro_f1"]) > 0.05),
    }
    metric_doc["governance"] = governance_results[candidate]
    metric_doc["already_tuned"] = False
    with metrics_path.open("w", encoding="utf-8") as fh:
        json.dump(metric_doc, fh, indent=2)
    print(f"Updated governance metadata for {candidate}.")


Updated governance metadata for hist_gradient_boosting_baseline.


## 6.2 Record exploratory Tier-2 candidate context

This cell records the dedicated Tier-2 exploratory context for `hist_gradient_boosting_baseline` in this notebook variant. It keeps candidate status, search outcomes, and governance-readiness metadata explicit so the run remains auditable and reproducible.

Inputs: candidate metadata, search outcomes, and output artifact checks.

Logic: preserve clear candidate-level context for downstream governance review and production handoff decisions.

Outputs: a documented candidate state aligned with PRD governance requirements.


In [20]:
production_candidates = [c for c, g in governance_results.items() if g["passed"]]
production_candidate = production_candidates[0] if production_candidates else next(iter(candidate_results.keys()))
if not governance_results[production_candidate]["passed"]:
    print(f"No tuned candidate passed governance; baseline stays provisional for {production_candidate}.")

for candidate, result in candidate_results.items():
    metrics_path = OUTPUT_DIR / f"{candidate}_tuned_metrics.json"
    with metrics_path.open("r", encoding="utf-8") as fh:
        metric_doc = json.load(fh)
    metric_doc["already_tuned"] = (
        bool(result["search"].best_params_)
        and (OUTPUT_DIR / f"{candidate}_tuned_model.pkl").exists()
        and (OUTPUT_DIR / f"{candidate}_tuned_metrics.json").exists()
        and (OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv").exists()
        and (OUTPUT_DIR / f"{candidate}_tuning_search_results.csv").exists()
    )
    with metrics_path.open("w", encoding="utf-8") as fh:
        json.dump(metric_doc, fh, indent=2)
    print(f"{candidate} already_tuned = {metric_doc['already_tuned']}")


No tuned candidate passed governance; baseline stays provisional for hist_gradient_boosting_baseline.
hist_gradient_boosting_baseline already_tuned = True


## 6.3 Persist search results and trial-level evidence

This cell exports the search results to CSV for each tuned candidate. It records the trial-level metrics, selected hyperparameters, and effective budget summary in a structured, parseable form. This evidence is essential for governance review and future scientific reproducibility.

Inputs: the completed search object and the per-candidate trial metadata.

Logic: serialize the trial results, best parameter set, and summary metrics to the required CSV naming convention.

Outputs: `*_tuning_search_results.csv` artifacts under `5_tune_model` and run-level search metadata.

In [21]:
manifest = {
    "model_name": production_candidate,
    "model_type": str(type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__module__) + "." + type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__name__,
    "model_path": str(Path(".") / "5_tune_model" / f"{production_candidate}_tuned_model.pkl"),
    "feature_columns": feature_cols,
    "label_mapping": {"buy": 0, "hold": 1, "sell": 2},
    "label_thresholds": {"buy": 0.05, "sell": -0.05},
    "prediction_horizon_days": 20,
    "scaler_required": False,
    "scaler_path": None,
    "training_data_end": train_val_df["date"].max().date().isoformat(),
    "test_metrics": tuned_test_payloads[production_candidate]["metrics"],
    "governance_status": governance_results[production_candidate],
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUTPUT_DIR / "production_model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved: {manifest_path}")


Manifest saved: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json


# 7. Final refit and one-time test evaluation

This section enforces the strict final-evaluation policy: after selecting the best hyperparameters, the notebook refits the model on the full train+validation data and evaluates it exactly once on the immutable hold-out test set. This is the core governance control that prevents repeated probing on the final test data.

In [22]:
required_artifacts = []
for candidate, payload in tuned_test_payloads.items():
    required_artifacts.extend([
        OUTPUT_DIR / f"{candidate}_tuned_model.pkl",
        OUTPUT_DIR / f"{candidate}_tuned_metrics.json",
        OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv",
        OUTPUT_DIR / f"{candidate}_tuning_search_results.csv",
    ])
required_artifacts.append(OUTPUT_DIR / "production_model_manifest.json")
missing = [str(p) for p in required_artifacts if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing artifact(s): {missing}")
for p in required_artifacts:
    print(f"Validated artifact: {p}")


Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_model.pkl
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_metrics.json
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_test_predictions.csv
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuning_search_results.csv
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json


## 7.1 Refit best candidate on train+validation

This cell refits the selected tuned model using the full train+validation subset after the best hyperparameters are chosen. It ensures that preprocessing is fit only within each training fold during CV, then the final fitted estimator is trained on the combined post-search data before final assessment.

Inputs: selected best parameters, the full train+validation data, and the pipeline or preprocessing configuration used during search.

Logic: compute the final fit, store the refit estimator, and keep explicit evidence that this is the single final model state before test evaluation.

Outputs: a final fitted model object ready for hold-out evaluation.

In [23]:
print("=== Tuning Run Summary ===")
for candidate, result in candidate_results.items():
    print(f"Candidate: {candidate}")
    print(f"  best_cv_score: {candidate_results[candidate]['best_score']:.4f}")
    print(f"  test_macro_f1: {tuned_test_payloads[candidate]['metrics']['macro_f1']:.4f}")
    print(f"  governance: {governance_results[candidate]}")
print(f"Selected production candidate: {production_candidate}")


=== Tuning Run Summary ===
Candidate: hist_gradient_boosting_baseline
  best_cv_score: 0.3670
  test_macro_f1: 0.3525
  governance: {'passed': False, 'reasons': ['fail_test_macro_f1'], 'flag_instability': False, 'val_macro_f1': 0.3669533569972454, 'test_macro_f1': 0.3525090899209624, 'test_f1_buy': 0.3227665706051873, 'test_f1_sell': 0.34574468085106386}
Selected production candidate: hist_gradient_boosting_baseline


## 7.2 Evaluate once on the immutable test set

This cell performs the final evaluation on the hold-out test set. It must compute the same metric schema used in baseline evaluation and record the modelâ€™s test predictions, metrics, and supporting metadata. It should guard against repeated test access by using a strict one-time evaluation policy.

Inputs: the final refit model and the immutable `X_test` / `y_test` pairs.

Logic: score the final model once, generate predictions, and store all metric outputs that will later be compared against baseline.

Outputs: test metrics, predictions CSV, and a final evaluation record for each tuned candidate.

In [24]:
final_eval_results = {}
for candidate, result in candidate_results.items():
    model = result["final_model"]
    y_pred = model.predict(X_test)
    final_eval_results[candidate] = {
        "actual": y_test.reset_index(drop=True).tolist(),
        "predicted": y_pred.tolist(),
        "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    }
    print(f"{candidate} final test macro F1: {final_eval_results[candidate]['macro_f1']:.4f}")


hist_gradient_boosting_baseline final test macro F1: 0.3525


## 7.3 Save tuned model, metrics, and predictions

This cell writes each tuned model artifact to disk in the projectâ€™s required naming convention. It saves the pickled model, the metrics JSON, and the test predictions CSV, keeping file names and schema consistent with the PRD. It should also maintain a canonical record of the artifacts that were created and whether they were produced successfully.

Inputs: final fitted model, test predictions, and final tuned metrics.

Logic: persist the artifacts under `5_tune_model` with the required filenames and versioned naming pattern.

Outputs: `*_tuned_model.pkl`, `*_tuned_metrics.json`, and `*_tuned_test_predictions.csv` for each candidate that completed the final evaluation.

In [25]:
for candidate, payload in tuned_test_payloads.items():
    model_path = OUTPUT_DIR / f"{candidate}_tuned_model.pkl"
    with model_path.open("wb") as fh:
        pickle.dump(candidate_results[candidate]["final_model"], fh)
    payload["predictions"].to_csv(OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv", index=False)
    print(f"Saved final artifacts for {candidate}: {model_path.name}")


Saved final artifacts for hist_gradient_boosting_baseline: hist_gradient_boosting_baseline_tuned_model.pkl


# 8. Baseline comparison and governance decision

This section compares the tuned candidate against the baseline model and applies the PRDâ€™s governance conditions. The notebook should compute the required lift metrics, classify instability based on validation-to-test drop, and preserve the baseline as provisional when a tuned candidate fails the pass rules.

In [26]:
comparison_rows = []
for candidate, payload in tuned_test_payloads.items():
    baseline = baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {})
    tuned = payload["metrics"]
    cov = candidate_results[candidate]["best_score"]
    row = {
        "model_name": candidate,
        "baseline_test_macro_f1": float(baseline.get("macro_f1", 0.0)),
        "tuned_test_macro_f1": float(tuned["macro_f1"]),
        "baseline_test_f1_buy": float(baseline.get("f1_buy", baseline.get("per_class", {}).get("buy", {}).get("f1", 0.0))),
        "tuned_test_f1_buy": float(tuned["f1_buy"]),
        "baseline_test_f1_sell": float(baseline.get("f1_sell", baseline.get("per_class", {}).get("sell", {}).get("f1", 0.0))),
        "tuned_test_f1_sell": float(tuned["f1_sell"]),
        "val_macro_f1": float(cov),
        "stability_delta": float(cov - tuned["macro_f1"]),
        "lift_macro_f1": float(tuned["macro_f1"] - baseline.get("macro_f1", 0.0)),
        "lift_f1_buy": float(tuned["f1_buy"] - float(baseline.get("f1_buy", baseline.get("per_class", {}).get("buy", {}).get("f1", 0.0)))),
        "lift_f1_sell": float(tuned["f1_sell"] - float(baseline.get("f1_sell", baseline.get("per_class", {}).get("sell", {}).get("f1", 0.0)))),
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = OUTPUT_DIR / "baseline_vs_tuned_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(comparison_df)


                        model_name  baseline_test_macro_f1  \
0  hist_gradient_boosting_baseline                  0.3688   

   tuned_test_macro_f1  baseline_test_f1_buy  tuned_test_f1_buy  \
0             0.352509                0.4661           0.322767   

   baseline_test_f1_sell  tuned_test_f1_sell  val_macro_f1  stability_delta  \
0                 0.3037            0.345745      0.366953         0.014444   

   lift_macro_f1  lift_f1_buy  lift_f1_sell  
0      -0.016291    -0.143333      0.042045  


## 8.1 Build the baseline-versus-tuned comparison table

This cell constructs the comparison table that shows baseline and tuned performance side by side for the relevant metrics. It must include at minimum `test_macro_f1`, `test_f1_buy`, `test_f1_sell`, and the validation-to-test stability delta required by governance.

Inputs: baseline metrics from step 3 and tuned metrics from the final evaluation.

Logic: compute the direct metric comparisons and the lifts from baseline to tuned performance.

Outputs: a comparison table and a structured comparison block that can be serialized to CSV or JSON.

In [27]:
governance_results = {}
for candidate, payload in tuned_test_payloads.items():
    baseline = baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {})
    tuned = payload["metrics"]
    val_macro_f1 = float(candidate_results[candidate]["best_score"])
    flag_instability = (val_macro_f1 - tuned["macro_f1"]) > 0.05
    reasons = []
    if tuned["macro_f1"] <= float(baseline.get("macro_f1", 0.0)):
        reasons.append("fail_test_macro_f1")
    if tuned["f1_buy"] < 0.20:
        reasons.append("fail_test_f1_buy")
    if tuned["f1_sell"] < 0.15:
        reasons.append("fail_test_f1_sell")
    if flag_instability:
        reasons.append("flag_instability")
    passed = len(reasons) == 0
    governance_results[candidate] = {
        "passed": passed,
        "reasons": reasons,
        "flag_instability": bool(flag_instability),
        "val_macro_f1": val_macro_f1,
        "test_macro_f1": tuned["macro_f1"],
        "test_f1_buy": tuned["f1_buy"],
        "test_f1_sell": tuned["f1_sell"],
    }
    print(f"{candidate}: {'PASS' if passed else 'FAIL'} | reasons={reasons}")


hist_gradient_boosting_baseline: FAIL | reasons=['fail_test_macro_f1']


## 8.2 Apply the governance pass/fail rules

This cell implements the exact pass/fail logic defined in the PRD. It checks whether the tuned candidate improves test macro F1 over the baseline, achieves the buy and sell F1 guardrails, and avoids the instability condition where validation-to-test macro F1 drop exceeds `0.05`. If the checks fail, the notebook should keep the baseline as the provisional candidate and document the failure reason in the output JSON.

Inputs: the tuned comparison values and the baseline metrics.

Logic: evaluate each governance condition, compute the instability flag, and classify the result as pass or fail.

Outputs: a governance result object and a failure reason list (if applicable).

In [28]:
for candidate, result in candidate_results.items():
    metrics_path = OUTPUT_DIR / f"{candidate}_tuned_metrics.json"
    with metrics_path.open("r", encoding="utf-8") as fh:
        metric_doc = json.load(fh)
    metric_doc["comparison"] = {
        "baseline_test_macro_f1": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("macro_f1", 0.0)),
        "tuned_test_macro_f1": float(tuned_test_payloads[candidate]["metrics"]["macro_f1"]),
        "lift_test_macro_f1": float(tuned_test_payloads[candidate]["metrics"]["macro_f1"] - baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("macro_f1", 0.0)),
        "baseline_test_f1_buy": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("f1_buy", baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("per_class", {}).get("buy", {}).get("f1", 0.0))),
        "tuned_test_f1_buy": float(tuned_test_payloads[candidate]["metrics"]["f1_buy"]),
        "baseline_test_f1_sell": float(baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("f1_sell", baseline_payload.get(candidate, {}).get("metrics", {}).get("test", {}).get("per_class", {}).get("sell", {}).get("f1", 0.0))),
        "tuned_test_f1_sell": float(tuned_test_payloads[candidate]["metrics"]["f1_sell"]),
        "stability_delta": float(candidate_results[candidate]["best_score"] - tuned_test_payloads[candidate]["metrics"]["macro_f1"]),
        "flag_instability": bool((candidate_results[candidate]["best_score"] - tuned_test_payloads[candidate]["metrics"]["macro_f1"]) > 0.05),
    }
    metric_doc["governance"] = governance_results[candidate]
    metric_doc["already_tuned"] = False
    with metrics_path.open("w", encoding="utf-8") as fh:
        json.dump(metric_doc, fh, indent=2)
    print(f"Updated governance metadata for {candidate}.")


Updated governance metadata for hist_gradient_boosting_baseline.


## 8.3 Set `already_tuned` status and document the final decision

This cell validates whether the run qualifies as a fully documented tuning result. It should set `already_tuned=true` only when systematic search metadata exists, the final refit occurred, the final test evaluation happened exactly once, and the required artifacts were saved. Otherwise the notebook should leave the status as false and explain why.

Inputs: governance state, model metadata, and artifact existence checks.

Logic: write a final candidate-level readiness statement for downstream production or governance review.

Outputs: a final tuning status object and a documented decision record.

In [29]:
production_candidates = [c for c, g in governance_results.items() if g["passed"]]
production_candidate = production_candidates[0] if production_candidates else next(iter(candidate_results.keys()))
if not governance_results[production_candidate]["passed"]:
    print(f"No tuned candidate passed governance; baseline remains provisional for {production_candidate}.")

for candidate, result in candidate_results.items():
    metrics_path = OUTPUT_DIR / f"{candidate}_tuned_metrics.json"
    with metrics_path.open("r", encoding="utf-8") as fh:
        metric_doc = json.load(fh)
    metric_doc["already_tuned"] = (
        bool(result["search"].best_params_)
        and (OUTPUT_DIR / f"{candidate}_tuned_model.pkl").exists()
        and (OUTPUT_DIR / f"{candidate}_tuned_metrics.json").exists()
        and (OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv").exists()
        and (OUTPUT_DIR / f"{candidate}_tuning_search_results.csv").exists()
    )
    with metrics_path.open("w", encoding="utf-8") as fh:
        json.dump(metric_doc, fh, indent=2)
    print(f"{candidate} already_tuned = {metric_doc['already_tuned']}")


No tuned candidate passed governance; baseline remains provisional for hist_gradient_boosting_baseline.
hist_gradient_boosting_baseline already_tuned = True


# 9. Artifact packaging and production handoff

This section packages all successful tuning outputs in a standardized way and prepares the production manifest required by the next notebook in the workflow. The notebook should ensure that the artifact paths are valid, project-relative, and consistent with the fixed schema defined in the PRD.

In [30]:
manifest = {
    "model_name": production_candidate,
    "model_type": str(type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__module__) + "." + type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__name__,
    "model_path": str(Path(".") / "5_tune_model" / f"{production_candidate}_tuned_model.pkl"),
    "feature_columns": feature_cols,
    "label_mapping": {"buy": 0, "hold": 1, "sell": 2},
    "label_thresholds": {"buy": 0.05, "sell": -0.05},
    "prediction_horizon_days": 20,
    "scaler_required": False,
    "scaler_path": None,
    "training_data_end": train_val_df["date"].max().date().isoformat(),
    "test_metrics": tuned_test_payloads[production_candidate]["metrics"],
    "governance_status": governance_results[production_candidate],
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUTPUT_DIR / "production_model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved: {manifest_path}")


Manifest saved: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json


## 9.1 Generate the production model manifest

This cell serializes the final production manifest for step 6. The manifest must include the fixed schema fields specified by the PRD: model name, model type, model path, feature columns, label mapping, label thresholds, prediction horizon, scaler requirements, training data end, test metrics, governance status, and created timestamp.

Inputs: the selected candidate, tuned model metadata, feature list, and the final governance decision.

Logic: build the manifest using only project-relative valid file paths and the exact required schema.

Outputs: `production_model_manifest.json` under `5_tune_model` and the final candidate handoff record.

In [31]:
manifest = {
    "model_name": production_candidate,
    "model_type": str(type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__module__) + "." + type(candidate_results[production_candidate]["final_model"].named_steps["model"]).__name__,
    "model_path": str(Path(".") / "5_tune_model" / f"{production_candidate}_tuned_model.pkl"),
    "feature_columns": feature_cols,
    "label_mapping": {"buy": 0, "hold": 1, "sell": 2},
    "label_thresholds": {"buy": 0.05, "sell": -0.05},
    "prediction_horizon_days": 20,
    "scaler_required": False,
    "scaler_path": None,
    "training_data_end": train_val_df["date"].max().date().isoformat(),
    "test_metrics": tuned_test_payloads[production_candidate]["metrics"],
    "governance_status": governance_results[production_candidate],
    "created_at": datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUTPUT_DIR / "production_model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved: {manifest_path}")


Manifest saved: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json


## 9.2 Validate artifact set and path safety

This cell performs a final validation pass on the files written by the notebook. It should confirm the expected artifact names exist under the approved directory, confirm that paths are project-relative and safe, and check that sensitive values are not being output to logs or notebooks.

Inputs: all tuned outputs and the final manifest.

Logic: validate existence, project-relative path safety, and artifact completeness before considering the notebook run successful.

Outputs: a validation summary and the final handoff status for downstream notebooks.

In [32]:
required_artifacts = []
for candidate, payload in tuned_test_payloads.items():
    required_artifacts.extend([
        OUTPUT_DIR / f"{candidate}_tuned_model.pkl",
        OUTPUT_DIR / f"{candidate}_tuned_metrics.json",
        OUTPUT_DIR / f"{candidate}_tuned_test_predictions.csv",
        OUTPUT_DIR / f"{candidate}_tuning_search_results.csv",
    ])
required_artifacts.append(OUTPUT_DIR / "production_model_manifest.json")
missing = [str(p) for p in required_artifacts if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing artifact(s): {missing}")
for p in required_artifacts:
    print(f"Validated artifact: {p}")


Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_model.pkl
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_metrics.json
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuned_test_predictions.csv
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/hist_gradient_boosting_baseline_tuning_search_results.csv
Validated artifact: /workspace/projects/1-month-direction-classifier/5_tune_model/production_model_manifest.json


# 10. Run summary and open questions

This final section summarizes the outcome of the HGB tuning run and highlights any open issues or ambiguous decisions that should be confirmed before production handoff. It should include a concise record of governance status and whether the notebook completed with all required artifacts and metadata.

Open question note: if the project requires a broader production selection policy across multiple tuned notebooks, this cell should capture that decision explicitly and state how the handoff logic would adapt.


In [33]:
print("=== Tuning Run Summary ===")
for candidate, result in candidate_results.items():
    print(f"Candidate: {candidate}")
    print(f"  best_cv_score: {candidate_results[candidate]['best_score']:.4f}")
    print(f"  test_macro_f1: {tuned_test_payloads[candidate]['metrics']['macro_f1']:.4f}")
    print(f"  governance: {governance_results[candidate]}")
print(f"Selected production candidate: {production_candidate}")


=== Tuning Run Summary ===
Candidate: hist_gradient_boosting_baseline
  best_cv_score: 0.3670
  test_macro_f1: 0.3525
  governance: {'passed': False, 'reasons': ['fail_test_macro_f1'], 'flag_instability': False, 'val_macro_f1': 0.3669533569972454, 'test_macro_f1': 0.3525090899209624, 'test_f1_buy': 0.3227665706051873, 'test_f1_sell': 0.34574468085106386}
Selected production candidate: hist_gradient_boosting_baseline
